In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Aufgabe 7 – Elektrisches Netzwerk

## (a) DGL aufstellen

Kirchhoffsche Knotenregel an **V₁** (Summe eingehender Ströme = 0):

$$\frac{U(t) - V_1}{R_1} - C_1 \dot{V}_1 - \frac{V_1 - V_2}{R_2} = 0$$

Kirchhoffsche Knotenregel an **V₂**:

$$\frac{V_1 - V_2}{R_2} - C_2 \dot{V}_2 = 0$$

Umgestellt ergibt sich $\mathbf{V}' = A\mathbf{V} + \mathbf{b}\sin(\varepsilon t)$ mit

$$A = \begin{pmatrix} -\frac{1}{C_1 R_1} - \frac{1}{C_1 R_2} & \frac{1}{C_1 R_2} \\ \frac{1}{C_2 R_2} & -\frac{1}{C_2 R_2} \end{pmatrix} = \begin{pmatrix} -1001 & 1000 \\ 1000 & -1000 \end{pmatrix}, \quad \mathbf{b} = \begin{pmatrix} 1 \\ 0 \end{pmatrix}$$

### Analytische Lösung

Charakteristisches Polynom von $A$: $\lambda^2 + 2001\lambda + 1000 = 0$, also

$$\lambda_{1,2} = \frac{-2001 \pm \sqrt{2001^2 - 4000}}{2} \approx -0.5,\ -2000.5$$

Ansatz für den partikulären Anteil: $\mathbf{V}_p = \boldsymbol{\alpha}\sin(\varepsilon t) + \boldsymbol{\beta}\cos(\varepsilon t)$. Einsetzen liefert:

$$A\boldsymbol{\alpha} + \varepsilon\boldsymbol{\beta} = -\mathbf{b}, \qquad A\boldsymbol{\beta} - \varepsilon\boldsymbol{\alpha} = \mathbf{0}$$

Allgemeine Lösung: $\mathbf{V}(t) = c_1 \boldsymbol{\phi}_1 e^{\lambda_1 t} + c_2 \boldsymbol{\phi}_2 e^{\lambda_2 t} + \mathbf{V}_p(t)$

Mit $V_1(0) = V_2(0) = 0$ bestimmt man $c_1, c_2$ aus $\Phi\mathbf{c} = -\boldsymbol{\beta}$.

In [ ]:
C1 = C2 = 0.001
R1, R2 = 1000, 1

A = np.array([
    [-1/(C1*R1) - 1/(C1*R2),  1/(C1*R2)],
    [ 1/(C2*R2),             -1/(C2*R2)]
])
b = np.array([1/(C1*R1), 0.0])

print("A =\n", A)
lam = np.linalg.eigvals(A)
print("Eigenwerte:", np.sort(lam.real)[::-1])
print("Steifigkeitsverhaeltnis:", max(abs(lam))/min(abs(lam)))

In [ ]:
def particular(eps):
    # A*alpha + eps*beta = -b
    # A*beta  - eps*alpha = 0
    M = np.block([[A, eps*np.eye(2)], [-eps*np.eye(2), A]])
    rhs = np.concatenate([-b, [0., 0.]])
    x = np.linalg.solve(M, rhs)
    return x[:2], x[2:]  # alpha, beta

def analytic(t, eps):
    # V(0) = [0, 0]
    eigvals, Phi = np.linalg.eig(A)
    alpha, beta = particular(eps)
    c = np.linalg.solve(Phi, -beta)
    exp_mat = c * np.exp(np.outer(t, eigvals))  # (N, 2)
    Vh = np.real((Phi @ exp_mat.T).T)
    Vp = np.outer(np.sin(eps*t), alpha) + np.outer(np.cos(eps*t), beta)
    return Vh + Vp

In [ ]:
def euler_exp(eps, h, T):
    t = np.arange(0, T + h/2, h)
    V = np.zeros((len(t), 2))
    for i in range(len(t)-1):
        V[i+1] = V[i] + h * (A @ V[i] + b * np.sin(eps * t[i]))
    return t, V

def euler_imp(eps, h, T):
    t = np.arange(0, T + h/2, h)
    V = np.zeros((len(t), 2))
    M_inv = np.linalg.inv(np.eye(2) - h * A)
    for i in range(len(t)-1):
        V[i+1] = M_inv @ (V[i] + h * b * np.sin(eps * t[i+1]))
    return t, V

In [ ]:
# --- eps = 1 ---
eps = 1
T   = 20

t_ref = np.linspace(0, T, 5000)
V_ref = analytic(t_ref, eps)

# Stabilitaetsgrenze explizit: h * |lambda_2| < 2  =>  h < 2/2000.5 ~ 0.001
te_ok,  Ve_ok  = euler_exp(eps, 0.001, T)
te_bad, Ve_bad = euler_exp(eps, 0.01,  T)

ti_grob, Vi_grob = euler_imp(eps, 0.1,  T)
ti_fein, Vi_fein = euler_imp(eps, 0.01, T)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(t_ref, V_ref[:,0], 'k-', lw=1.2, label='analytisch')
ax.plot(te_ok, Ve_ok[:,0], 'r--', lw=0.9, label='explizit h=0.001')
# instabiler Fall: nur solange die Werte noch im sichtbaren Bereich sind
ylim = ax.get_ylim()
safe = np.isfinite(Ve_bad[:,0]) & (abs(Ve_bad[:,0]) < 1)
ax.plot(te_bad[safe], Ve_bad[safe,0], 'g--', lw=0.9, label='explizit h=0.01 (instabil)')
ax.set_title('ε = 1 – explizit')
ax.set_xlabel('t')
ax.set_ylabel('V₁(t)')
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(t_ref, V_ref[:,0], 'k-', lw=1.2, label='analytisch')
ax.plot(ti_grob, Vi_grob[:,0], 'b--', lw=0.9, label='implizit h=0.1')
ax.plot(ti_fein, Vi_fein[:,0], 'm--', lw=0.9, label='implizit h=0.01')
ax.set_title('ε = 1 – implizit')
ax.set_xlabel('t')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# --- eps = 1000 ---
# Periode der Schwingung: 2*pi/1000 ~ 0.0063
# Stabilitaetsgrenze explizit unveraendert: h < 0.001
# => explizit loest Schwingung gerade noch auf (ca. 7 Punkte/Periode)
# => implizit benoetigt h < Periode fuer gute Genauigkeit, sonst wird Oszillation gedaempft
eps = 1000
T   = 0.05

t_ref = np.linspace(0, T, 10000)
V_ref = analytic(t_ref, eps)

te_ok, Ve_ok = euler_exp(eps, 0.0009, T)

ti_fein, Vi_fein = euler_imp(eps, 0.0001, T)
ti_grob, Vi_grob = euler_imp(eps, 0.002,  T)  # zu grob fuer Schwingung

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(t_ref, V_ref[:,0], 'k-', lw=1, label='analytisch')
ax.plot(te_ok, Ve_ok[:,0], 'r--', lw=0.9, label='explizit h=0.0009')
ax.set_title('ε = 1000 – explizit')
ax.set_xlabel('t')
ax.set_ylabel('V₁(t)')
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(t_ref, V_ref[:,0], 'k-', lw=1, label='analytisch')
ax.plot(ti_fein, Vi_fein[:,0], 'b--', lw=0.9, label='implizit h=0.0001')
ax.plot(ti_grob, Vi_grob[:,0], 'm--', lw=0.9, label='implizit h=0.002 (Schwingung verloren)')
ax.set_title('ε = 1000 – implizit')
ax.set_xlabel('t')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## (b) Diskussion – Steifigkeit

Die Eigenwerte von $A$ sind $\lambda_1 \approx -0.5$ und $\lambda_2 \approx -2000.5$. Das **Steifigkeitsverhältnis** beträgt $|\lambda_2/\lambda_1| \approx 4001$ → das System ist **steif**.

**Explizites Eulerverfahren:** Die Stabilitätsbedingung $h|\lambda_2| < 2$ fordert $h < 0.001$, unabhängig von $\varepsilon$. Für $\varepsilon = 1$ ist diese Schrittweite viel feiner als nötig, um die langsame Lösung aufzulösen – man zahlt einen hohen Preis für die Steifigkeit.

**Implizites Eulerverfahren:** Unbedingt stabil. Für $\varepsilon = 1$ reicht $h = 0.1$. Für $\varepsilon = 1000$ muss $h$ die Schwingungsperiode $2\pi/1000 \approx 0.006$ auflösen, sonst wird die Oszillation weggedämpft – hier begrenzt die Genauigkeit, nicht die Stabilität.

**Fazit:** Die DGL ist steif. Das implizite Eulerverfahren ist für steife Systeme deutlich effizienter.